# 2.2 Q1 — Why fixed-size chunking breaks on lecture transcripts

Fixed-size chunking cuts text every N tokens, regardless of what is being said. On edited writing like blog posts this is often tolerable: the text is dense, punctuated, and organised into paragraphs that each carry one idea, so a blind cut rarely destroys much meaning.

Raw lecture transcripts violate every one of those assumptions:

- **Filler words** ("um", "uh", "you know", repeated words) consume tokens without carrying meaning. A 512-token window of speech holds less actual content than 512 tokens of written text, and the filler dilutes the chunk's embedding.
- **Missing punctuation** means there are no reliable sentence boundaries, so cuts land mid-thought, and even simple "cut at the end of a sentence" fallbacks aren't available.
- **Variable speech pace** means a fixed token count maps to an inconsistent amount of meaning: 512 tokens might cover three ideas from a fast speaker or half an idea from a slow one.
- **ASR errors** can corrupt the key terms themselves (e.g. "exhaust port" → "exhaust sport"), so the chunk containing the answer may not even contain the query's keywords.

The deeper problem is how people *speak*. A lecturer introduces a topic, digresses, and only then explains it, usually referring back with "it" or "this" instead of repeating the name. The introduction and the explanation end up in different chunks, and the explanation chunk becomes **orphan context**: its meaning depends on text stored elsewhere. Retrieval then finds the introduction (which matches the query's keywords) and misses the explanation (which doesn't).

The experiments below reproduce this failure on a small scale:

In [1]:
import sys
print(sys.executable)
import tiktoken
print("tiktoken OK")

/home/sabari/.venv/bin/python3
tiktoken OK


## Tokens are not words

Chunk sizes are measured in tokens, not words. Common English words are usually a single token, but rare words, names, and numbers get split into several. So the amount of *text* that fits in a 512-token window depends on the vocabulary being used; a jargon-heavy lecture fits fewer words per chunk than a casual one.

In [3]:
examples = ["so it is a small opening",
            "Pourbaix diagrams in Chennai at 42.7 degrees"]

for text in examples:
    n_words = len(text.split())
    n_tokens = len(enc.encode(text))
    print(f"{n_words} words -> {n_tokens} tokens | {text}")

6 words -> 6 tokens | so it is a small opening
7 words -> 12 tokens | Pourbaix diagrams in Chennai at 42.7 degrees


## Fixed-size chunking on a messy transcript

Below is a short synthetic transcript designed to mimic real lecture speech: no punctuation, filler words, a repeated word, a digression, and an explanation that refers to the topic only as "it". I chunk it with a fixed window of 20 tokens, a scaled-down stand-in for 512 tokens on a full lecture.

In [4]:
transcript = ("um so today we are going to look at the thermal exhaust port "
              "uh which is you know one of the the key weak points and "
              "before i explain why let me just recall what we did last week "
              "so it is a small opening and it leads straight to the core")
tokens = enc.encode(transcript)

def fixed_chunks(tokens, chunk_size, overlap=0):
    step = chunk_size - overlap
    chunks = []
    for i in range(0, len(tokens), step):
        chunks.append(enc.decode(tokens[i:i + chunk_size]))
    return chunks

for i, c in enumerate(fixed_chunks(tokens, chunk_size=20)):
    print(f"--- chunk {i} ---\n{c}")

--- chunk 0 ---
um so today we are going to look at the thermal exhaust port uh which is you know one of
--- chunk 1 ---
 the the key weak points and before i explain why let me just recall what we did last week so
--- chunk 2 ---
 it is a small opening and it leads straight to the core


### What went wrong

- **Chunk 0** names the topic ("thermal exhaust port") but is cut mid-phrase ("one of"); it contains no explanation.
- **Chunk 1** mentions "key weak points" but never says *what* is weak; the rest is a digression.
- **Chunk 2** contains the actual explanation ("it is a small opening... leads straight to the core") but never names the topic.

For the query *"What are the key vulnerabilities of the thermal exhaust port?"*, chunk 0 matches best on keywords but holds no answer, while chunk 2, the only one with the answer, shares almost no words with the query. This is exactly the failure described in the question.

In [5]:
for overlap in [0, 5, 15]:
    chunks = fixed_chunks(tokens, chunk_size=20, overlap=overlap)
    rescued = any("thermal exhaust port" in c and "small opening" in c
                  for c in chunks)
    print(f"overlap={overlap:2d} | chunks={len(chunks):2d} | "
          f"topic + explanation in same chunk: {rescued}")

overlap= 0 | chunks= 3 | topic + explanation in same chunk: False
overlap= 5 | chunks= 4 | topic + explanation in same chunk: False
overlap=15 | chunks=11 | topic + explanation in same chunk: False


## Does overlap fix it?

The standard patch for boundary problems is overlap: each chunk repeats the last few tokens of the previous one. I tested overlaps of 0, 5, and 15 tokens and checked automatically whether any chunk contained both the topic and its explanation.

**Result:** no amount of overlap rescued the explanation, while the number of chunks grew from 3 to 11.

The reason is structural. In this transcript, "thermal exhaust port" and "small opening" are roughly 30+ tokens apart, but each chunk is only 20 tokens long. **No chunk can contain both ends of a gap larger than the chunk itself**, regardless of overlap. Overlap can only rescue context that is closer together than the chunk size. In real lectures, a digression between introducing a topic and explaining it can last minutes, far beyond any reasonable overlap.

Meanwhile, overlap has real costs at archive scale (~40,000 hours):

- **Storage and compute:** at 75% overlap, the archive produces several times as many chunks to embed and store.
- **Retrieval crowding:** near-duplicate chunks can fill the top-k results with the same passage shifted by a few tokens, pushing out other relevant content.
- **Junk fragments:** trailing chunks can be tiny, meaningless scraps that still get embedded and retrieved.

**Conclusion:** chunk boundaries for spoken transcripts need to be decided by meaning or structure, not by token count. That motivates the strategies compared in Q2.